# Transform Refunds Data
1. Extract specific portion of the string from refund_reason using split function
2. Extract specific portion of the string from refund_reason using regexp_extract function
3. Write transformed data to the Silver schema in hive metastore

## 1. Extract specific portion of the string from refund_reason using split function

In [0]:
df = spark.table("hive_metastore.bronze.refunds")
display(df)

In [0]:
from pyspark.sql import functions as F
df_split_refunds = (
    df.select(
        "refund_id",
        "payment_id",
        "refund_timestamp",
        "refund_amount",
        F.split("refund_reason", ":")[0].alias("refund_reason"),
        F.split("refund_reason", ":")[1].alias("refund_source"),
    )
)

display(df_split_refunds)

## 2. Extract specific portion of the string from refund_reason using regexp_extract function

In [0]:
 
df_transformed_refunds = (
    df.select(
        "refund_id",
        "payment_id",
        F.date_format("refund_timestamp", "yyyy-MM-dd").alias("refund_date"),
        F.date_format("refund_timestamp", "HH:mm:ss").alias("refund_time"),
        "refund_amount",
        F.regexp_extract("refund_reason", "^([^:]+):", 1).alias("refund_reason"),
        F.regexp_extract("refund_reason", "^[^:]+:(.*)$", 1).alias("refund_source"),
    )
)

display(df_transformed_refunds)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hive_metastore.silver;

## 3. Write transformed data to the Silver schema in hive metastore

In [0]:
df_transformed_refunds.writeTo("gizmobox.silver.py_refunds").createOrReplace()

In [0]:
df = spark.table("gizmobox.silver.py_refunds")
display(df)